# F1 Race Strategy — Exploratory Data Analysis\n\n2023 British Grand Prix race lap data loaded from `data/processed/2023_british_gp_race_laps.csv`.

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

CSV_PATH = Path("../data/processed/2023_british_gp_race_laps.csv")

df = pd.read_csv(CSV_PATH)

# LapTime arrives as a timedelta string like "0 days 00:01:33.433000"
# Convert to total seconds for numeric analysis
df["LapTimeSec"] = pd.to_timedelta(df["LapTime"]).dt.total_seconds()

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nDtypes:\n", df.dtypes)
print("\nNull counts:\n", df.isnull().sum())

Shape: (971, 10)

Columns: ['LapNumber', 'Driver', 'Team', 'LapTime', 'Compound', 'TyreLife', 'PitOutTime', 'PitInTime', 'IsAccurate', 'LapTimeSec']

Dtypes:
 LapNumber     float64
Driver         object
Team           object
LapTime        object
Compound       object
TyreLife      float64
PitOutTime     object
PitInTime      object
IsAccurate       bool
LapTimeSec    float64
dtype: object

Null counts:
 LapNumber       0
Driver          0
Team            0
LapTime         5
Compound        0
TyreLife        0
PitOutTime    947
PitInTime     945
IsAccurate      0
LapTimeSec      5
dtype: int64


## 1 — Lap Time Progression per Driver

In [2]:
accurate = df[df["IsAccurate"] == True].copy()

fig = px.line(
    accurate.sort_values("LapNumber"),
    x="LapNumber",
    y="LapTimeSec",
    color="Driver",
    title="Lap Time Progression per Driver — 2023 British GP",
    labels={"LapTimeSec": "Lap Time (s)", "LapNumber": "Lap"},
    hover_data=["Compound", "TyreLife"],
)
fig.update_layout(height=550, legend_title="Driver")
fig.show()

## 2 — Tyre Degradation: Average Lap Time vs Tyre Age by Compound

In [3]:
deg = (
    accurate.groupby(["Compound", "TyreLife"], as_index=False)["LapTimeSec"]
    .mean()
    .rename(columns={"LapTimeSec": "AvgLapTimeSec"})
)

compound_colors = {"SOFT": "red", "MEDIUM": "gold", "HARD": "lightgrey", "INTERMEDIATE": "green", "WET": "blue"}

fig = px.line(
    deg.sort_values("TyreLife"),
    x="TyreLife",
    y="AvgLapTimeSec",
    color="Compound",
    color_discrete_map=compound_colors,
    markers=True,
    title="Tyre Degradation: Avg Lap Time vs Tyre Age — 2023 British GP",
    labels={"AvgLapTimeSec": "Avg Lap Time (s)", "TyreLife": "Tyre Age (laps)"},
)
fig.update_layout(height=480)
fig.show()

## 3 — Pit Stop Distribution: Which Laps Pit Stops Happened On

In [4]:
# A pit stop occurred on laps where PitInTime is not null
pit_laps = df[df["PitInTime"].notna()][["LapNumber", "Driver", "Compound"]].copy()

fig = px.histogram(
    pit_laps,
    x="LapNumber",
    color="Compound",
    color_discrete_map=compound_colors,
    nbins=52,
    title="Pit Stop Distribution by Lap — 2023 British GP",
    labels={"LapNumber": "Lap", "count": "Number of Pit Stops"},
    barmode="stack",
)
fig.update_layout(height=420, yaxis_title="Pit Stops")
fig.show()

print(f"Total pit stops: {len(pit_laps)}")
print(pit_laps.groupby("Compound").size().rename("stops"))

Total pit stops: 26
Compound
HARD       5
MEDIUM    15
SOFT       6
Name: stops, dtype: int64


## 4 — Lap Time Distribution per Team

In [5]:
team_order = (
    accurate.groupby("Team")["LapTimeSec"].median().sort_values().index.tolist()
)

fig = px.box(
    accurate,
    x="Team",
    y="LapTimeSec",
    color="Team",
    category_orders={"Team": team_order},
    title="Lap Time Distribution per Team — 2023 British GP",
    labels={"LapTimeSec": "Lap Time (s)", "Team": "Constructor"},
    points=False,
)
fig.update_layout(height=500, showlegend=False)
fig.update_xaxes(tickangle=30)
fig.show()